# Stage inputs before running decisions

## Goal

Sometimes a decision's inputs arrive at different times. An order may arrive first, while the customer information needed to evaluate it comes later. A temporary store lets those producers contribute data before a decision job starts.

Rulebricks calls its named state stores **contexts**. Each store holds records by a business key. Here, we stage order amounts, add country information later, then run the flow once the inputs are ready.

| Point in the pipeline | Stored inputs for one request |
| --- | --- |
| Order arrives | `request_id`, `amount` |
| Country arrives later | `request_id`, `amount`, `country` |
| Decision job starts | Read the assembled inputs and execute the flow |

This is input staging: temporary storage before use. Use it when data needs to be available inside Rulebricks across processing steps or producers. If the job already has every input in one Spark table, [bulk flows](02-bulk-flows.ipynb) is the simpler starting point.

## Setup

Follow the [installation guide](../docs/getting-started.md). This example uses a flow named `order-routing-context-flow`: it accepts `request_id`, reads the amount and country from **Spark staged requests**, then runs the order-routing rule. Edit the API URL, published version, context name, and output path below to match your workspace.

The API key also needs context creation, write, and read permissions. Automatic decision triggering stays off so this job controls when the flow runs.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, functions
from pyspark.sql.types import StructType

from rulebricks_spark import apply_flow, contexts

spark = (
    SparkSession.getActiveSession() or SparkSession.builder.appName("input-staging").getOrCreate()
)
api_key = os.environ["RULEBRICKS_API_KEY"]
base_url = "https://rulebricks.acme.com/api/v1"
flow_slug = "order-routing-context-flow"
flow_version = "1"
context_name = "Spark staged requests"
output_root = "file:///tmp/rulebricks-examples"
run_id = str(uuid.uuid4())
run_path = f"{output_root}/{run_id}"

requests = spark.createDataFrame(
    [(f"{run_id}-{i}", float(i * 10)) for i in range(20)],
    "request_id string, amount double",
)
countries = spark.createDataFrame(
    [(f"{run_id}-{i}", "US") for i in range(20)],
    "request_id string, country string",
)

## 1. Prepare the temporary store

`ensure()` creates the named context if it is missing, or checks that an existing one matches. The schema lists the fields producers may supply. The `request_id` key keeps each request independent.

| Concept | In this example |
| --- | --- |
| Context definition | One named store with three possible input fields |
| Context instance | The stored facts for one `request_id` |
| Retention | One day; choose a window that covers your own pipeline |

Creating the definition does not create records. Submitting data creates missing instances automatically.

In [ ]:
context_slug = contexts.ensure(
    context_name,
    schema=StructType()
    .add("request_id", "string")
    .add("amount", "double")
    .add("country", "string"),
    key="request_id",
    ttl_seconds=86400,
    history_limit=1,
    api_key=api_key,
    base_url=base_url,
)
print("Context:", context_slug)

## 2. Stage the first inputs

Only the amounts have arrived. `ingest()` stores them and returns a **receipt**, which tells us what happened to the write. It does not return a business decision.

| Receipt table column | Meaning |
| --- | --- |
| `request_id`, `amount` | Submitted inputs |
| `rulebricks_result` | JSON receipt describing acceptance and any related activity |
| `rulebricks_meta` | Write outcome and errors |

Save and check the receipts before proceeding.

In [ ]:
staged = contexts.ingest(
    requests,
    context_slug,
    key="request_id",
    api_key=api_key,
    base_url=base_url,
)
staged.write.format("delta").mode("errorifexists").save(f"{run_path}/amount-receipts")
amount_receipts = spark.read.format("delta").load(f"{run_path}/amount-receipts")
assert amount_receipts.count() == 20
assert amount_receipts.filter("rulebricks_meta.status != 'ok'").count() == 0

## 3. Add information from a later source

The second submission uses the same request IDs and supplies only `country`. Rulebricks merges those fields into each stored record, retaining its amount.

| Before | New submission | After |
| --- | --- | --- |
| `amount=50` | `country="US"` | `amount=50`, `country="US"` |

In a real pipeline, these two submissions could come from different jobs. Here they run together so the full example is easy to follow.

In [ ]:
enriched = contexts.ingest(
    countries,
    context_slug,
    key="request_id",
    api_key=api_key,
    base_url=base_url,
)
enriched.write.format("delta").mode("errorifexists").save(f"{run_path}/country-receipts")
country_receipts = spark.read.format("delta").load(f"{run_path}/country-receipts")
assert country_receipts.count() == 20
assert country_receipts.filter("rulebricks_meta.status != 'ok'").count() == 0

## 4. Check that the inputs have arrived

`lookup()` reads the stored state by key. Here we check that both producers have finished and that every request has an amount and a country.

The saved lookup is an observation. The flow will read the live context itself, so producers should finish their updates before this decision job starts. Keeping the reference data unchanged also makes this read-only workload safe to repeat.

This job knows which fields it needs and checks them explicitly. Automatic readiness-based triggering is a separate context capability.


In [ ]:
observed = contexts.lookup(
    requests.select("request_id"),
    context_slug,
    key="request_id",
    execution_partitions=2,
    api_key=api_key,
    base_url=base_url,
)
observed.write.format("delta").mode("errorifexists").save(f"{run_path}/input-snapshot")
snapshot = spark.read.format("delta").load(f"{run_path}/input-snapshot")
assert snapshot.filter("rulebricks_meta.status != 'ok'").count() == 0

ready = snapshot.select(
    "request_id",
    functions.get_json_object("rulebricks_result", "$.state.amount").cast("double").alias("amount"),
    functions.get_json_object("rulebricks_result", "$.state.country").alias("country"),
)
assert ready.count() == 20
assert ready.filter("amount IS NULL OR country IS NULL").count() == 0
assert ready.select("request_id", "amount").exceptAll(requests).count() == 0
ready.show(5, truncate=False)

## 5. Send only the request IDs

Each flow reads its amount and country from the corresponding context instance. Spark sends just `request_id`, keeping the bulk payload small even when the staged records contain much more data.

The result retains that request ID alongside the flow response and execution metadata.


In [ ]:
evaluated = apply_flow(
    ready.select("request_id"),
    flow_slug,
    version=flow_version,
    correlation_id_path="request_id",
    execution_partitions=2,
    api_key=api_key,
    base_url=base_url,
)
evaluated.write.format("delta").mode("errorifexists").save(f"{run_path}/decisions")
saved = spark.read.format("delta").load(f"{run_path}/decisions")
assert saved.count() == 20
assert saved.filter("rulebricks_meta.status != 'ok'").count() == 0
saved.select("request_id", "rulebricks_result").show(5, truncate=False)

## Next steps

Keep the input snapshot, write receipts, and decisions with the pipeline run. Contexts are bounded, expiring operational stores; your data lake or warehouse remains the durable source of record.

A repeated key merges fields into one record. It does not append rows to a table. Assign one key per staged request, and coordinate producers that might overwrite the same field. After an uncertain write, inspect receipts and stored state before resubmitting.

When each event must use the result of the previous one, continue with [stateful processing](05-stateful-processing.ipynb). For context-triggered decisions and ordering details, see the [context guide](../docs/contexts.md).